In [3]:
# ==========================================
# 홈플레이트 프레임 단위 탐지율 측정 스크립트
# Google Colab 환경용
# ==========================================

# 1. 패키지 설치
!pip install ultralytics

# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import cv2
import os
import csv
import time
import unicodedata
from pathlib import Path
from ultralytics import YOLO

# ==========================================
# [경로 설정] - 기존 평가 스크립트와 동일 구조
# ==========================================
DRIVE_ROOT = Path("/content/drive/MyDrive")

# 홈플레이트 모델
HOMEPLATE_MODEL_PATH = str(DRIVE_ROOT / "newhomeplate.pt")

# 영상 폴더 (편집된 체크스윙 영상들)
VIDEO_DIR = DRIVE_ROOT / "KBO_edit"

# 결과 CSV
OUTPUT_CSV = DRIVE_ROOT / "homeplate_detection_rate.csv"

# ==========================================
# 홈플레이트 탐지율 측정
# ==========================================
def measure_homeplate_detection(video_path, model):
    """영상의 전체 프레임에서 홈플레이트 탐지율을 계산"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, 0

    total_frames = 0
    detected_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        total_frames += 1

        # 홈플레이트 모델 추론 (conf=0.3, 기존 평가와 동일)
        results = model(frame, conf=0.3, verbose=False)

        # 키포인트가 유효하게 감지되었는지 확인
        if (results[0].keypoints is not None
            and len(results[0].keypoints) > 0):
            kpts = results[0].keypoints[0].xy.cpu().numpy()[0]
            # 최소 4개 키포인트가 유효한 좌표(>1)를 가져야 탐지 성공
            if (len(kpts) >= 4
                and kpts[0][0] > 1
                and kpts[1][0] > 1
                and kpts[3][0] > 1):
                detected_frames += 1

    cap.release()
    return detected_frames, total_frames


def main():
    print("홈플레이트 모델 로딩 중...")
    model = YOLO(HOMEPLATE_MODEL_PATH)

    video_files = sorted([
        f for f in VIDEO_DIR.iterdir()
        if f.suffix.lower() == ".mp4"
    ])
    total_videos = len(video_files)
    print(f"총 {total_videos}개 영상에서 홈플레이트 탐지율 측정 시작...\n")

    # CSV 헤더
    fieldnames = ['filename', 'detected_frames', 'total_frames', 'detection_rate']
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    all_detected = 0
    all_total = 0

    for i, video_file in enumerate(video_files):
        start_t = time.time()
        detected, total = measure_homeplate_detection(video_file, model)
        rate = (detected / total * 100) if total > 0 else 0

        all_detected += detected
        all_total += total

        elapsed = time.time() - start_t
        print(f"[{i+1}/{total_videos}] {video_file.name} | "
              f"{detected}/{total} frames ({rate:.1f}%) | {elapsed:.1f}s")

        # CSV에 영상별 결과 추가
        row = {
            'filename': video_file.name,
            'detected_frames': detected,
            'total_frames': total,
            'detection_rate': f"{rate:.2f}%"
        }
        with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8-sig') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

    # 최종 결과 출력
    overall_rate = (all_detected / all_total * 100) if all_total > 0 else 0
    print(f"\n{'='*50}")
    print(f"[최종 결과] 홈플레이트 프레임 단위 탐지율")
    print(f"{'='*50}")
    print(f"전체 프레임: {all_total}")
    print(f"탐지 프레임: {all_detected}")
    print(f"평균 탐지율: {overall_rate:.2f}%")
    print(f"{'='*50}")
    print(f"\n결과 CSV: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
홈플레이트 모델 로딩 중...
총 145개 영상에서 홈플레이트 탐지율 측정 시작...

[1/145] 250819_SSGKT_8초_노스윙.mp4 | 97/97 frames (100.0%) | 7.8s
[2/145] 250820_SSGKT_8말_노스윙.mp4 | 186/186 frames (100.0%) | 6.0s
[3/145] 250820_롯데LG_9초_노스윙.mp4 | 7/60 frames (11.7%) | 3.0s
[4/145] 250820_삼성NC_9초_노스윙.mp4 | 167/167 frames (100.0%) | 5.6s
[5/145] 250820_키움KIA_7초_스윙.mp4 | 144/144 frames (100.0%) | 4.9s
[6/145] 250821_SSGKT_5말_노스윙.mp4 | 62/62 frames (100.0%) | 4.1s
[7/145] 250821_SSGKT_9말_스윙.mp4 | 3/132 frames (2.3%) | 4.6s
[8/145] 250821_롯데LG_7말_스윙.mp4 | 74/74 frames (100.0%) | 3.2s
[9/145] 250821_삼성NC_8초_스윙.mp4 | 186/186 frames (100.0%) | 5.7s
[10/145] 250821_키움KIA_4말_노스윙.mp4 | 61/61 frames (100.0%) | 3.1s
[11/145] 250822_KT두산_8초_스윙.mp4 | 76/96 frames (79.2%) | 3.8s
[12/145] 250822_LGKIA_1말_노스윙.mp4 | 118/118 frames 

In [6]:
# ==========================================
# 배트(Bat) 키포인트 모델 프레임 단위 탐지율 측정
# Google Colab 환경용
# ==========================================

import cv2
import os
import csv
import time
from pathlib import Path
from ultralytics import YOLO

# 1. 경로 설정 (홈플레이트 때 성공했던 구글 드라이브 원본 경로로 롤백!)
DRIVE_ROOT = Path("/content/drive/MyDrive")
BAT_MODEL_PATH = str(DRIVE_ROOT / "best_3.pt")  # 배트 키포인트 모델
VIDEO_DIR = DRIVE_ROOT / "KBO_edit"             # 구글 드라이브 안의 원본 영상 폴더
OUTPUT_CSV = DRIVE_ROOT / "bat_pose_detection_rate.csv"

def measure_bat_pose_detection(video_path, model):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, 0

    total_frames = 0
    detected_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        total_frames += 1

        # 배트 모델 추론 (device=0으로 A100 GPU 가동)
        results = model(frame, conf=0.3, verbose=False, device=0)

        # 키포인트가 유효하게 감지되었는지 확인
        if (results[0].keypoints is not None and len(results[0].keypoints) > 0):
            # xy 좌표 추출
            kpts = results[0].keypoints[0].xy.cpu().numpy()[0]

            # 배트의 유효한 키포인트 개수 세기 (좌표가 1보다 큰 정상적인 점)
            valid_kpts = [pt for pt in kpts if pt[0] > 1 and pt[1] > 1]

            # 배트 점이 2개 이상 정상 인식되면 탐지 성공으로 간주 (기본값 2)
            if len(valid_kpts) >= 2:
                detected_frames += 1

    cap.release()
    return detected_frames, total_frames

def main():
    print("배트 키포인트 모델 로딩 중...")
    model = YOLO(BAT_MODEL_PATH)

    video_files = sorted([
        f for f in VIDEO_DIR.iterdir()
        if f.suffix.lower() == ".mp4"
    ])
    total_videos = len(video_files)
    print(f"총 {total_videos}개 영상에서 배트 탐지율 측정 시작 (A100 GPU)...\n")

    # CSV 헤더
    fieldnames = ['filename', 'detected_frames', 'total_frames', 'detection_rate']
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    all_detected = 0
    all_total = 0

    for i, video_file in enumerate(video_files):
        start_t = time.time()
        detected, total = measure_bat_pose_detection(video_file, model)
        rate = (detected / total * 100) if total > 0 else 0

        all_detected += detected
        all_total += total

        elapsed = time.time() - start_t
        print(f"[{i+1}/{total_videos}] {video_file.name} | "
              f"{detected}/{total} frames ({rate:.1f}%) | {elapsed:.1f}s")

        # CSV에 영상별 결과 추가
        row = {
            'filename': video_file.name,
            'detected_frames': detected,
            'total_frames': total,
            'detection_rate': f"{rate:.2f}%"
        }
        with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8-sig') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

    # 최종 결과 출력
    overall_rate = (all_detected / all_total * 100) if all_total > 0 else 0
    print(f"\n{'='*50}")
    print(f"[최종 결과] 배트(Pose) 프레임 단위 탐지율")
    print(f"{'='*50}")
    print(f"전체 프레임: {all_total}")
    print(f"탐지 프레임: {all_detected}")
    print(f"평균 탐지율: {overall_rate:.2f}%")
    print(f"{'='*50}")
    print(f"\n결과 CSV: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

배트 키포인트 모델 로딩 중...
총 145개 영상에서 배트 탐지율 측정 시작 (A100 GPU)...

[1/145] 250819_SSGKT_8초_노스윙.mp4 | 97/97 frames (100.0%) | 2.5s
[2/145] 250820_SSGKT_8말_노스윙.mp4 | 184/186 frames (98.9%) | 4.3s
[3/145] 250820_롯데LG_9초_노스윙.mp4 | 51/60 frames (85.0%) | 1.4s
[4/145] 250820_삼성NC_9초_노스윙.mp4 | 164/167 frames (98.2%) | 4.0s
[5/145] 250820_키움KIA_7초_스윙.mp4 | 142/144 frames (98.6%) | 3.5s
[6/145] 250821_SSGKT_5말_노스윙.mp4 | 45/62 frames (72.6%) | 1.6s
[7/145] 250821_SSGKT_9말_스윙.mp4 | 98/132 frames (74.2%) | 3.1s
[8/145] 250821_롯데LG_7말_스윙.mp4 | 74/74 frames (100.0%) | 1.8s
[9/145] 250821_삼성NC_8초_스윙.mp4 | 183/186 frames (98.4%) | 4.5s
[10/145] 250821_키움KIA_4말_노스윙.mp4 | 56/61 frames (91.8%) | 1.5s
[11/145] 250822_KT두산_8초_스윙.mp4 | 90/96 frames (93.8%) | 2.1s
[12/145] 250822_LGKIA_1말_노스윙.mp4 | 118/118 frames (100.0%) | 2.6s
[13/145] 250822_LGKIA_4초_스윙.mp4 | 103/107 frames (96.3%) | 2.4s
[14/145] 250822_SSG한화_11초_스윙.mp

In [7]:
import cv2
from ultralytics import YOLO
from pathlib import Path

# 경로 설정
DRIVE_ROOT = Path("/content/drive/MyDrive")
BAT_MODEL_PATH = str(DRIVE_ROOT / "best_3.pt")
HOMEPLATE_MODEL_PATH = str(DRIVE_ROOT / "newhomeplate.pt")

# 아무 영상이나 하나 샘플로 지정 (경로에 있는 첫 번째 영상 사용)
SAMPLE_VIDEO = str(DRIVE_ROOT / "KBO_edit" / "250819_SSGKT_8초_노스윙.mp4")

def measure_pure_inference_speed():
    print("모델 로딩 및 A100 GPU 웜업 중... (약 10초 소요)")
    bat_model = YOLO(BAT_MODEL_PATH)
    hp_model = YOLO(HOMEPLATE_MODEL_PATH)

    # 샘플 프레임 1장 추출
    cap = cv2.VideoCapture(SAMPLE_VIDEO)
    ret, frame = cap.read()
    cap.release()

    # GPU 웜업 (처음 추론할 때 발생하는 지연 시간 제거)
    for _ in range(5):
        bat_model(frame, device=0, verbose=False)
        hp_model(frame, device=0, verbose=False)

    # 실제 속도 측정 (10번 반복 후 평균값 계산)
    bat_speeds, hp_speeds = [], []
    for _ in range(10):
        res_bat = bat_model(frame, device=0, verbose=False)
        res_hp = hp_model(frame, device=0, verbose=False)
        bat_speeds.append(res_bat[0].speed['inference'])
        hp_speeds.append(res_hp[0].speed['inference'])

    avg_bat_ms = sum(bat_speeds) / len(bat_speeds)
    avg_hp_ms = sum(hp_speeds) / len(hp_speeds)
    total_ms = avg_bat_ms + avg_hp_ms
    total_fps = 1000 / total_ms

    print("\n" + "="*50)
    print("🚀 NVIDIA A100 기반 순수 추론 속도 측정 결과")
    print("="*50)
    print(f"- 배트 Pose 모델 추론 속도: {avg_bat_ms:.2f} ms/frame")
    print(f"- 홈플레이트 Pose 모델 추론 속도: {avg_hp_ms:.2f} ms/frame")
    print(f"- 듀얼 모델 통합 추론 속도: 합계 {total_ms:.2f} ms/frame")
    print(f"👉 예상 처리 프레임률(FPS): 약 {total_fps:.1f} FPS")
    print("="*50)

measure_pure_inference_speed()

모델 로딩 및 A100 GPU 웜업 중... (약 10초 소요)

🚀 NVIDIA A100 기반 순수 추론 속도 측정 결과
- 배트 Pose 모델 추론 속도: 10.49 ms/frame
- 홈플레이트 Pose 모델 추론 속도: 10.46 ms/frame
- 듀얼 모델 통합 추론 속도: 합계 20.95 ms/frame
👉 예상 처리 프레임률(FPS): 약 47.7 FPS
